In [1]:
import random
import time
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical
from IPython.display import clear_output
from google.colab import drive

drive.mount('/content/drive')


# =========================================================
# 1. ENV
# =========================================================

class TagMARLEnv:
    """
    1 Pacman vs 2 Ghosts
    Partial observability with ray radar + MAPPO global critic state

    Reward:
      - If ghost i can see pacman this step: ghost i gets +1
      - If any ghost can see pacman this step: pacman gets -1
      - else 0

    Actor obs:
      - Pacman: 8 rays * 5 = 40
      - Ghost : 8 rays * 9 = 72

    Critic obs:
      - global state = 190
    """

    ACTIONS = {
        0: (-1, 0),   # UP
        1: (1, 0),    # DOWN
        2: (0, -1),   # LEFT
        3: (0, 1),    # RIGHT
        4: (0, 0),    # STAY
    }

    ACTION_NAMES = {
        0: "UP",
        1: "DOWN",
        2: "LEFT",
        3: "RIGHT",
        4: "STAY",
    }

    RAYS = [
        (-1, 0),   # N
        (-1, 1),   # NE
        (0, 1),    # E
        (1, 1),    # SE
        (1, 0),    # S
        (1, -1),   # SW
        (0, -1),   # W
        (-1, -1),  # NW
    ]

    MAP_LIBRARY = {
        "empty": [
            "###########",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "###########",
        ],
        "medium": [
            "###########",
            "#000000000#",
            "#000#00000#",
            "#000#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00000#",
            "#000000000#",
            "###########",
        ],
        "hard": [
            "###########",
            "#0000#0000#",
            "#0##0#0##0#",
            "#0000#0000#",
            "###0000#0##",
            "#000##0000#",
            "#0#000##00#",
            "#000#000#0#",
            "#0##0#0000#",
            "#000000000#",
            "###########",
        ],
    }

    def __init__(
        self,
        max_steps=400,
        seed=42,
        min_start_dist=4,
        pacman_view_range=10,
        ghost_view_range=3,
        map_name="medium",
    ):
        self.max_steps = max_steps
        self.rng = random.Random(seed)
        self.min_start_dist = min_start_dist

        self.pacman_speed = 1
        self.ghost_speed = 1

        self.pacman_view_range = pacman_view_range
        self.ghost_view_range = ghost_view_range

        if map_name not in self.MAP_LIBRARY:
            raise ValueError(f"Unknown map_name: {map_name}. Choose from {list(self.MAP_LIBRARY.keys())}")

        self.raw_map = self.MAP_LIBRARY[map_name]
        self.map_name = map_name

        self.H = len(self.raw_map)
        self.W = len(self.raw_map[0])

        self.empty_cells = [
            (r, c)
            for r in range(self.H)
            for c in range(self.W)
            if self.raw_map[r][c] == "0"
        ]

        self.reset()

    def reset(self):
        self.grid = [list(row) for row in self.raw_map]

        while True:
            sampled_positions = self.rng.sample(self.empty_cells, 3)
            pacman = sampled_positions[0]
            ghosts = sampled_positions[1:]

            if all(self.manhattan(pacman, g) >= self.min_start_dist for g in ghosts):
                self.pacman = pacman
                self.ghosts = ghosts
                break

        self.steps = 0
        self.done = False
        return self.get_obs()

    def in_bounds(self, r, c):
        return 0 <= r < self.H and 0 <= c < self.W

    def move(self, pos, action):
        dr, dc = self.ACTIONS[action]
        nr, nc = pos[0] + dr, pos[1] + dc

        if not self.in_bounds(nr, nc):
            return pos
        if self.grid[nr][nc] == "#":
            return pos
        return (nr, nc)

    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _ray_scan_for_ghosts(self, start_pos, view_range):
        sx, sy = start_pos
        feat = []
        ghost_set = set(self.ghosts)

        for dr, dc in self.RAYS:
            wall_dist = 0
            target_found = False
            target_dist = 0
            target_dx = 0
            target_dy = 0
            target_mask = 0

            r, c = sx, sy
            for step in range(1, view_range + 1):
                r += dr
                c += dc

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    wall_dist = step
                    break

                wall_dist = step

                if not target_found and (r, c) in ghost_set:
                    target_found = True
                    target_dist = step
                    target_dx = r - sx
                    target_dy = c - sy
                    target_mask = 1
            else:
                wall_dist = view_range + 1

            feat.extend([
                wall_dist / (view_range + 1),
                target_dist / max(1, view_range),
                target_dx / self.H,
                target_dy / self.W,
                float(target_mask),
            ])

        return np.array(feat, dtype=np.float32)

    def _ray_scan_ghost_dual(self, ghost_idx, view_range):
        sx, sy = self.ghosts[ghost_idx]
        ally_set = set(
            self.ghosts[j] for j in range(len(self.ghosts)) if j != ghost_idx
        )

        feat = []

        for dr, dc in self.RAYS:
            wall_dist = 0

            pacman_found = False
            pacman_dist = 0
            pacman_dx = 0
            pacman_dy = 0
            pacman_mask = 0

            ally_found = False
            ally_dist = 0
            ally_dx = 0
            ally_dy = 0
            ally_mask = 0

            r, c = sx, sy
            for step in range(1, view_range + 1):
                r += dr
                c += dc

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    wall_dist = step
                    break

                wall_dist = step

                if not pacman_found and (r, c) == self.pacman:
                    pacman_found = True
                    pacman_dist = step
                    pacman_dx = self.pacman[0] - sx
                    pacman_dy = self.pacman[1] - sy
                    pacman_mask = 1

                if not ally_found and (r, c) in ally_set:
                    ally_found = True
                    ally_dist = step
                    ally_dx = r - sx
                    ally_dy = c - sy
                    ally_mask = 1
            else:
                wall_dist = view_range + 1

            feat.extend([
                wall_dist / (view_range + 1),

                pacman_dist / max(1, view_range),
                pacman_dx / self.H,
                pacman_dy / self.W,
                float(pacman_mask),

                ally_dist / max(1, view_range),
                ally_dx / self.H,
                ally_dy / self.W,
                float(ally_mask),
            ])

        return np.array(feat, dtype=np.float32)

    def ghost_can_see_pacman(self, ghost_pos):
        gx, gy = ghost_pos

        for dr, dc in self.RAYS:
            r, c = gx, gy
            for _ in range(self.ghost_view_range):
                r += dr
                c += dc

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break

                if (r, c) == self.pacman:
                    return True

        return False

    def pacman_obs(self):
        return self._ray_scan_for_ghosts(self.pacman, self.pacman_view_range)

    def ghost_obs_single(self, idx):
        return self._ray_scan_ghost_dual(idx, self.ghost_view_range)

    def global_state(self):
        px, py = self.pacman
        coords = [px / self.H, py / self.W]

        for gx, gy in self.ghosts:
            coords.extend([gx / self.H, gy / self.W])

        pac_actor = self.pacman_obs()
        ghost_actor = np.stack(
            [self.ghost_obs_single(i) for i in range(len(self.ghosts))],
            axis=0
        )

        gstate = np.concatenate([
            np.array(coords, dtype=np.float32),
            pac_actor,
            ghost_actor.reshape(-1),
        ], axis=0)

        return gstate.astype(np.float32)

    def get_obs(self):
        pacman_actor = self.pacman_obs()
        ghost_actor = np.stack(
            [self.ghost_obs_single(i) for i in range(len(self.ghosts))],
            axis=0
        )

        gstate = self.global_state()

        pacman_critic = gstate.copy()
        ghost_critic = np.stack([gstate.copy() for _ in range(len(self.ghosts))], axis=0)

        return {
            "pacman_actor": pacman_actor,
            "pacman_critic": pacman_critic,
            "ghost_actor": ghost_actor,
            "ghost_critic": ghost_critic,
        }

    def step(self, pacman_action, ghost_actions):
        if self.done:
            raise RuntimeError("Episode ended. Call reset().")

        self.steps += 1

        self.pacman = self.move(self.pacman, pacman_action)
        self.ghosts = [self.move(g, a) for g, a in zip(self.ghosts, ghost_actions)]

        ghost_seen_flags = np.array(
            [1.0 if self.ghost_can_see_pacman(g) else 0.0 for g in self.ghosts],
            dtype=np.float32
        )

        pacman_reward = -1.0 if ghost_seen_flags.sum() > 0 else 0.0
        ghost_rewards = ghost_seen_flags.copy()

        if self.steps >= self.max_steps:
            self.done = True

        obs = self.get_obs()
        rewards = {
            "pacman": pacman_reward,
            "ghosts": ghost_rewards
        }
        dones = {
            "pacman": float(self.done),
            "ghosts": np.array([float(self.done)] * len(self.ghosts), dtype=np.float32)
        }
        return obs, rewards, dones, {}

    def render_text(self):
        board = [row[:] for row in self.grid]
        pr, pc = self.pacman
        board[pr][pc] = "P"
        for i, (gr, gc) in enumerate(self.ghosts):
            board[gr][gc] = str(i + 1)
        print("\n".join("".join(row) for row in board))
        print(f"map={self.map_name}, steps={self.steps}, done={self.done}")


# =========================================================
# 2. MODEL
# =========================================================

class RecurrentActorCritic(nn.Module):
    def __init__(self, actor_obs_dim, critic_obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.actor_encoder = nn.Sequential(
            nn.Linear(actor_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.critic_encoder = nn.Sequential(
            nn.Linear(critic_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.actor_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)
        self.critic_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)

        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def init_hidden(self, batch_size, device):
        actor_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        actor_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        return actor_h, actor_c, critic_h, critic_c

    def forward_step(self, actor_obs, critic_obs, hidden):
        actor_h, actor_c, critic_h, critic_c = hidden

        actor_x = self.actor_encoder(actor_obs).unsqueeze(0)
        critic_x = self.critic_encoder(critic_obs).unsqueeze(0)

        actor_out, (actor_h, actor_c) = self.actor_lstm(actor_x, (actor_h, actor_c))
        critic_out, (critic_h, critic_c) = self.critic_lstm(critic_x, (critic_h, critic_c))

        actor_out = actor_out.squeeze(0)
        critic_out = critic_out.squeeze(0)

        logits = self.policy_head(actor_out)
        value = self.value_head(critic_out).squeeze(-1)

        hidden = (actor_h, actor_c, critic_h, critic_c)
        return logits, value, hidden


# =========================================================
# 3. CHECKPOINT LOAD
# =========================================================

def load_models(checkpoint_path, device="cpu", hidden_dim=128):
    pacman_actor_dim = 40
    ghost_actor_dim = 72
    critic_obs_dim = 190

    pacman_net = RecurrentActorCritic(
        actor_obs_dim=pacman_actor_dim,
        critic_obs_dim=critic_obs_dim,
        action_dim=5,
        hidden_dim=hidden_dim
    ).to(device)

    ghost_net = RecurrentActorCritic(
        actor_obs_dim=ghost_actor_dim,
        critic_obs_dim=critic_obs_dim,
        action_dim=5,
        hidden_dim=hidden_dim
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)

    pacman_net.load_state_dict(ckpt["pacman_model_state_dict"])
    ghost_net.load_state_dict(ckpt["ghost_model_state_dict"])

    pacman_net.eval()
    ghost_net.eval()

    print(f"[Loaded checkpoint] update={ckpt.get('update', 'unknown')}")
    return pacman_net, ghost_net, ckpt


# =========================================================
# 4. INFERENCE / PLAY
# =========================================================

@torch.no_grad()
def run_inference(
    checkpoint_path,
    map_name="medium",
    seed=42,
    max_steps=400,
    pacman_view_range=4,
    ghost_view_range=3,
    device=None,
    stochastic=True,
    render=True,
    delay=0.3,
    clear_screen=True,
    hidden_dim=128,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    env = TagMARLEnv(
        max_steps=max_steps,
        seed=seed,
        min_start_dist=4,
        pacman_view_range=pacman_view_range,
        ghost_view_range=ghost_view_range,
        map_name=map_name,
    )

    pacman_net, ghost_net, ckpt = load_models(
        checkpoint_path=checkpoint_path,
        device=device,
        hidden_dim=hidden_dim
    )

    obs = env.reset()

    pac_hidden = pacman_net.init_hidden(1, device)
    ghost_hidden = ghost_net.init_hidden(len(env.ghosts), device)

    pac_total = 0.0
    ghost_total = np.zeros(len(env.ghosts), dtype=np.float32)
    done = False
    step = 0

    while not done:
        pac_actor_obs = torch.tensor(
            obs["pacman_actor"], dtype=torch.float32, device=device
        ).unsqueeze(0)
        pac_critic_obs = torch.tensor(
            obs["pacman_critic"], dtype=torch.float32, device=device
        ).unsqueeze(0)

        ghost_actor_obs = torch.tensor(
            obs["ghost_actor"], dtype=torch.float32, device=device
        )
        ghost_critic_obs = torch.tensor(
            obs["ghost_critic"], dtype=torch.float32, device=device
        )

        pac_logits, _, pac_hidden = pacman_net.forward_step(
            pac_actor_obs, pac_critic_obs, pac_hidden
        )
        ghost_logits, _, ghost_hidden = ghost_net.forward_step(
            ghost_actor_obs, ghost_critic_obs, ghost_hidden
        )

        if stochastic:
            pac_dist = Categorical(logits=pac_logits)
            pac_action = pac_dist.sample().item()

            ghost_dist = Categorical(logits=ghost_logits)
            ghost_actions = ghost_dist.sample().cpu().numpy().tolist()
        else:
            pac_action = torch.argmax(pac_logits, dim=-1).item()
            ghost_actions = torch.argmax(ghost_logits, dim=-1).cpu().numpy().tolist()

        obs, rewards, dones, _ = env.step(pac_action, ghost_actions)

        pac_total += rewards["pacman"]
        ghost_total += rewards["ghosts"]
        done = bool(dones["pacman"])

        if render:
            if clear_screen:
                clear_output(wait=True)

            print(f"[Checkpoint update] {ckpt.get('update', 'unknown')}")
            print(f"[Step] {step}")
            print(f"Pacman action: {env.ACTION_NAMES[pac_action]}")
            print(f"Ghost actions: {[env.ACTION_NAMES[a] for a in ghost_actions]}")
            print()

            env.render_text()
            print("Pacman reward:", rewards["pacman"])
            print("Ghost rewards:", rewards["ghosts"])
            print("Pacman total:", pac_total)
            print("Ghost mean total:", ghost_total.mean())
            print("-" * 40)

            time.sleep(delay)

        step += 1

    print("\n=== Final Result ===")
    print("Pacman total reward:", pac_total)
    print("Ghost mean total reward:", ghost_total.mean())
    print("Ghost total reward vector:", ghost_total)


Mounted at /content/drive


In [4]:
# =========================================================
# 5. RUN
# =========================================================

CHECKPOINT_PATH = "/content/drive/MyDrive/tag_mappo_checkpoints/checkpoint_update_1000.pt"

run_inference(
    checkpoint_path=CHECKPOINT_PATH,
    map_name="medium",      # "empty", "medium", "hard"
    seed=451,
    max_steps=400,
    pacman_view_range=4,
    ghost_view_range=3,
    device="cuda" if torch.cuda.is_available() else "cpu",
    stochastic=True,        # False면 argmax
    render=True,
    delay=0.1,
    clear_screen=True,
    hidden_dim=128,
)

[Checkpoint update] 1000
[Step] 399
Pacman action: LEFT
Ghost actions: ['DOWN', 'DOWN']

###########
#000000000#
#000#00000#
#000#00#00#
#0#0#00#00#
#0#0#00#00#
#0#0#00#00#
#0#0#00#00#
#0#0#00020#
#000000P00#
###########
map=medium, steps=400, done=True
Pacman reward: -1.0
Ghost rewards: [1. 1.]
Pacman total: -360.0
Ghost mean total: 293.0
----------------------------------------

=== Final Result ===
Pacman total reward: -360.0
Ghost mean total reward: 293.0
Ghost total reward vector: [293. 293.]
